In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd
import os
def relabel_csv_simple(csv_path, label_dict, output_path=None):
    """
    Gán nhãn lại CSV - chỉ nhãn 0 và 1
    label_dict: {(tile_id, image_id): 0 hoặc 1}
    """
    print(f"Đang đọc: {csv_path}")
    df = pd.read_csv(csv_path)

    if 'Tile_ID' not in df.columns or 'Image_ID' not in df.columns or 'LABEL' not in df.columns:
        print("CSV thiếu cột cần thiết")
        return None
    changed_count = 0
    changed_tiles = []

    for (tile_id, image_id), new_label in label_dict.items():
        if new_label not in [0, 1]:
            print(f"Tile {tile_id}: Nhãn {new_label} không hợp lệ (chỉ 0 hoặc 1)")
            continue
        mask = (df['Tile_ID'] == tile_id) & (df['Image_ID'] == image_id)
        if mask.any():
            old_label = df.loc[mask, 'LABEL'].iloc[0]
            if old_label != new_label:
                df.loc[mask, 'LABEL'] = new_label
                count = mask.sum()
                changed_count += count
                changed_tiles.append((tile_id, image_id, old_label, new_label, count))
                print(f"Tile {tile_id}: {old_label} → {new_label} ({count} dòng)")
    if output_path is None:
        base, ext = os.path.splitext(csv_path)
        output_path = f"{base}_relabeled{ext}"

    df.to_csv(output_path, index=False)

    print(f"\nKẾT QUẢ:")
    print(f"   File mới: {output_path}")
    print(f"   Tổng dòng: {len(df):,}")
    print(f"   Dòng thay đổi: {changed_count:,}")
    print(f"   Tiles thay đổi: {len(changed_tiles)}")

    print(f"\n PHÂN BỐ NHÃN:")
    zero_count = (df['LABEL'] == 0).sum()
    one_count = (df['LABEL'] == 1).sum()
    print(f"   Nhãn 0 (không mây): {zero_count:,} ({zero_count/len(df)*100:.1f}%)")
    print(f"   Nhãn 1 (mây): {one_count:,} ({one_count/len(df)*100:.1f}%)")

    return df, output_path, changed_tiles


def create_mask_image(rgb_path, label, output_path):
    """
    Tạo ảnh mask từ ảnh RGB và nhãn
    label = 0: không mask (ảnh gốc)
    label = 1: toàn bộ mask đỏ
    """
    from PIL import Image
    import numpy as np

    # Mở ảnh RGB
    rgb_img = Image.open(rgb_path)
    rgb_array = np.array(rgb_img)

    if label == 0:
        # Không mask
        result = rgb_array
    elif label == 1:
        # Toàn bộ mask đỏ
        height, width = rgb_array.shape[:2]
        mask = np.ones((height, width), dtype=np.uint8)

        # Màu đỏ, độ trong 50%
        red = np.array([255, 0, 0], dtype=np.uint8)
        mask_3d = np.stack([mask, mask, mask], axis=-1)
        red_mask = np.full_like(rgb_array, red)

        result = np.where(
            mask_3d > 0,
            (rgb_array * 0.5 + red_mask * 0.5).astype(np.uint8),
            rgb_array
        )
    else:
        print(f" Nhãn {label} không hợp lệ")
        return None

    # Lưu ảnh
    Image.fromarray(result).save(output_path)
    return output_path


def update_images_for_relabeled_tiles(changed_tiles, rgb_dir, mask_dir):
    """
    Tạo lại ảnh mask cho các tile đã thay đổi nhãn
    """
    if not os.path.exists(rgb_dir):
        print(f" Không tìm thấy thư mục RGB: {rgb_dir}")
        return

    os.makedirs(mask_dir, exist_ok=True)

    print(f"\nTạo ảnh mask mới...")

    for tile_id, image_id, old_label, new_label, count in changed_tiles:
        # Tìm ảnh RGB
        base_name = os.path.splitext(image_id)[0]
        rgb_pattern = f"*{base_name}*tile_{tile_id:04d}*.png"

        rgb_found = None
        for file in os.listdir(rgb_dir):
            if f"tile_{tile_id:04d}" in file and base_name in file and "_MASKED" not in file:
                rgb_found = os.path.join(rgb_dir, file)
                break

        if not rgb_found:
            print(f"Không tìm thấy ảnh RGB cho Tile {tile_id}")
            continue

        # Tạo tên file mask mới
        mask_filename = f"{base_name}_tile_{tile_id:04d}_MASKED_label{new_label}.png"
        mask_path = os.path.join(mask_dir, mask_filename)

        # Tạo ảnh mask
        try:
            create_mask_image(rgb_found, new_label, mask_path)
            print(f"Tile {tile_id}: Đã tạo mask label {new_label}")
        except Exception as e:
            print(f"Lỗi tạo mask Tile {tile_id}: {e}")

# ----------------------------------------------------
# 4. HÀM CHÍNH - TẤT CẢ TRONG 1
# ----------------------------------------------------

def relabel_and_update(csv_path, label_dict, rgb_dir=None, mask_dir=None):
    """
    Gán nhãn CSV và cập nhật ảnh - chỉ nhãn 0 và 1
    """
    print("=" * 50)
    print("GÁN NHÃN LẠI (CHỈ 0 VÀ 1)")
    print("=" * 50)

    # 1. Gán nhãn CSV
    df, new_csv, changed = relabel_csv_simple(csv_path, label_dict)

    if df is None:
        return

    # 2. Cập nhật ảnh nếu có thư mục
    if changed and rgb_dir and mask_dir:
        update_images_for_relabeled_tiles(changed, rgb_dir, mask_dir)

    print("\nHOÀN THÀNH!")
    return df, new_csv


In [ ]:
if __name__ == "__main__":
    CSV_FILE = "/content/drive/MyDrive/GEE S2_Training_Data/Dask_Split_by_Image/S2_149_TRAINING_data.csv"
    RGB_DIR = "/content/drive/MyDrive/GEE S2_Training_Data/RGB_Tiles"
    MASK_DIR = "/content/drive/MyDrive/GEE S2_Training_Data/RGB_Cloud_Mask_Tiles"

    NEW_LABELS = {
        (81, "S2_149_TRAINING.tif"): 0,
        (87, "S2_149_TRAINING.tif"): 0,
         (92, "S2_149_TRAINING.tif"): 0,
         (93, "S2_149_TRAINING.tif"): 0,
         (107, "S2_149_TRAINING.tif"): 0,

    }

    result = relabel_and_update(CSV_FILE, NEW_LABELS, RGB_DIR, MASK_DIR)

In [2]:
import pandas as pd
import os


def delete_from_csv(csv_path, items_to_delete, output_path=None):

    print(f" Đọc file: {csv_path}")
    df = pd.read_csv(csv_path)
    original_len = len(df)

    delete_mask = pd.Series([False] * len(df))

    for item in items_to_delete:
        if isinstance(item, tuple):
            tile_id, image_id = item
            mask = (df['Tile_ID'] == tile_id) & (df['Image_ID'] == image_id)
            count = mask.sum()
            if count > 0:
                print(f" Xóa Tile {tile_id} ({image_id}): {count:,} dòng")

        elif isinstance(item, int):
            tile_id = item
            mask = df['Tile_ID'] == tile_id
            count = mask.sum()
            if count > 0:
                print(f" Xóa Tile {tile_id} (tất cả): {count:,} dòng")

        elif isinstance(item, str):
            image_id = item
            mask = df['Image_ID'] == image_id
            count = mask.sum()
            if count > 0:
                tiles_count = df[mask]['Tile_ID'].nunique()
                print(f" Xóa Image '{image_id}': {count:,} dòng, {tiles_count} tiles")

        else:
            print(f"  Bỏ qua item không hợp lệ: {item}")
            continue

        delete_mask = delete_mask | mask

    # Xóa dữ liệu
    df_clean = df[~delete_mask].copy()
    deleted_count = delete_mask.sum()

    # Tạo tên file mới
    if output_path is None:
        base, ext = os.path.splitext(csv_path)
        output_path = f"{base}_cleaned{ext}"

    # Lưu file
    df_clean.to_csv(output_path, index=False)

    # Thống kê
    print(f"\n KẾT QUẢ:")
    print(f"   File gốc: {original_len:,} dòng")
    print(f"   Đã xóa: {deleted_count:,} dòng")
    print(f"   Còn lại: {len(df_clean):,} dòng")
    print(f"   File mới: {output_path}")

    if len(df_clean) > 0:
        zero = (df_clean['LABEL'] == 0).sum()
        one = (df_clean['LABEL'] == 1).sum()
        total = len(df_clean)
        print(f"\n Phân bố nhãn:")
        print(f"   Nhãn 0: {zero:,} ({zero/total*100:.1f}%)")
        print(f"   Nhãn 1: {one:,} ({one/total*100:.1f}%)")

    return df_clean, output_path

# ----------------------------------------------------
# 2. HÀM CHÍNH - CHỈ XÓA CSV
# ----------------------------------------------------

def clean_csv_only(csv_path, items_to_delete):
    """
    Chỉ xóa dữ liệu trong CSV, không động đến ảnh
    """
    print("=" * 50)
    print("XÓA DỮ LIỆU TRONG CSV")
    print("=" * 50)

    return delete_from_csv(csv_path, items_to_delete)



In [ ]:

if __name__ == "__main__":
    CSV_FILE = "/content/drive/MyDrive/GEE S2_Training_Data/S2_TRAINING_DATA_FILTERED_TILES_COORDS_V2.csv"
    ITEMS_TO_DELETE = [
        (5, "S2_117_TRAINING.tif"),
         (6, "S2_117_TRAINING.tif"),
         (5, "S2_117_TRAINING.tif"),
         (5, "S2_117_TRAINING.tif"),
         (5, "S2_117_TRAINING.tif"),
          (5, "S2_117_TRAINING.tif"),

          ]

    # 2. Xóa theo tile_id
    # ITEMS_TO_DELETE = [1, 2, 3]  # xóa tile 1, 2, 3

    # 3. Xóa theo image_id (toàn bộ file)
    # ITEMS_TO_DELETE = ["S2_117_TRAINING.tif"]  # xóa toàn bộ file này

    df_clean, new_file = clean_csv_only(CSV_FILE, ITEMS_TO_DELETE)

    print(f"\n XONG!")